[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/02.%20The%20Container%20Stacking%20Rules%20Problem/P2-Tier-4_executed.ipynb)

---


# 2. The Container Stacking Rules Problem

## Tier 4 — A reinforcement-learning approach

Earlier tiers construct a stacking plan by solving or searching for one. This tier demonstrates a different approach: learning a policy that selects a stack from the current observed state.

The notebook trains a linear action-value model with Q-learning updates and an experience-replay buffer. It uses only NumPy, pandas, and Matplotlib; it does not use a neural-network framework. The method is DQN-inspired, but it is not a canonical Deep Q-Network: it has a linear function approximator and no target network.

### Learning goals

- Follow the reinforcement-learning interaction loop: state, action, reward, and next state.
- Understand how blocking-pair risk, height imbalance, and a simple weight-instability signal enter the reward.
- Trace epsilon-greedy action selection, replay-buffer sampling, and temporal-difference updates.
- Interpret training diagnostics and evaluate a greedy policy against an exact small-instance benchmark.

### Inputs and outputs

The input is a fixed set of containers, their departure times and weights, a fixed arrival order, and stack-capacity limits. The notebook outputs training curves, a greedy policy run, a stacking visualization, and an exact comparison of blocking-pair counts for this small instance.

### Scope and modeling assumptions

Containers arrive in a deterministic order, and the agent places each container on top of one stack. The environment does not model retrieval operations, travel distances, crane scheduling, or other operational costs. A blocking pair is used as a proxy for reshuffle risk: the count is the number of lower–upper container pairs for which the lower container departs first. This pair count is not necessarily the number of physical reshuffle moves in every operational retrieval procedure.

The agent observes a compact feature vector rather than the complete stack contents. This representation can merge distinct physical configurations, so it is an approximation and is not guaranteed to be a Markov state. Training and evaluation use the same fixed instance; this experiment alone does not establish generalization to other arrival orders or container data.

In [ ]:
# Check that the notebook's required packages are available; do not install packages here.
# A managed Jupyter environment should provide these dependencies in advance.

from typing import List, Dict, Tuple, Optional

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in your Python environment."
    ) from e

# A single seeded generator makes exploration, initialization, and replay sampling reproducible.
rng = np.random.default_rng(0)

print("Dependencies imported successfully.")

## Problem setup: a small environment

The instance contains five containers, three stacks, and at most two tiers per stack. The arrival order is fixed. At time step \(t\), the agent chooses a stack for the arriving container, which is placed on top of that stack. The episode ends after all containers have been placed.

The reward is the negative of a weighted penalty. The penalty encourages the agent to avoid creating blocking pairs, producing imbalanced stack heights, and placing a heavier container directly on top of a lighter one. A terminal penalty also accounts for the final blocking-pair count. These are modeling signals for learning, not a complete operational cost model.

### Placement and penalty definitions

A placement decision changes one stack at a time. Let \(S_t=(S_{1,t},\ldots,S_{N,t})\) denote the stack configuration before placing the next container, with each stack listed from bottom to top. Let \(a_t\in\{1,\ldots,N\}\) be the selected stack, and let \(c_t\) be the container arriving at time \(t\). The action is feasible when stack \(a_t\) has fewer than \(M\) containers. The transition appends \(c_t\) to that stack and leaves the other stacks unchanged.

For a stack \(S=(c_1,\ldots,c_h)\), the blocking-pair proxy is

$$B(S)=\sum_{1\leq i<j\leq h}\mathbf{1}\{d_{c_i}<d_{c_j}\},$$

where \(c_i\) is below \(c_j\), \(d_c\) is container \(c\)'s departure time, and \(\mathbf{1}\{\cdot\}\) is 1 when its condition is true and 0 otherwise. A lower container departing earlier than an upper container contributes one pair: the upper container must be removed first if the lower one is to be retrieved. Equal departure times do not contribute under this strict comparison.

When container \(c_t\) is placed on stack \(a_t\), the newly created blocking-pair count is

$$b_t=\sum_{c\in S_{a_t,t}}\mathbf{1}\{d_c<d_{c_t}\}.$$

The stack-height imbalance after placement is measured by population variance. If \(h_{i,t+1}=|S_{i,t+1}|\) is the height of stack \(i\), then

$$V_{t+1}=\frac{1}{N}\sum_{i=1}^{N}(h_{i,t+1}-\bar h_{t+1})^2,
\qquad
\bar h_{t+1}=\frac{1}{N}\sum_{i=1}^{N}h_{i,t+1}.$$

The local weight-instability signal is \(u_t=1\) when the arriving container is heavier than the current top container, and \(u_t=0\) otherwise; an empty stack has signal 0. With nonnegative penalty weights \(w_b,w_v,w_w\), the immediate reward is

$$r_t=-(w_b b_t+w_v V_{t+1}+w_w u_t).$$

On the final placement, the environment additionally subtracts \(\lambda B(S_{T})\), where \(B(S_T)\) is the sum of blocking-pair counts across all final stacks and \(\lambda\) is the terminal reshuffle weight. Thus the local pair penalty and terminal pair penalty are both part of the specified learning reward. The code implements these definitions in `blocking_created_by_placing`, `height_variance`, `weight_instability_created`, and `step_env`.

In [ ]:
# Example instance: departure time d and weight w are attributes of each container.
containers = [
    {"id": "C1", "d": 3, "w": 22},
    {"id": "C2", "d": 1, "w": 20},
    {"id": "C3", "d": 5, "w": 18},
    {"id": "C4", "d": 2, "w": 25},
    {"id": "C5", "d": 6, "w": 16},
]

NUM_STACKS = 3
MAX_TIERS = 2

cid_to_d = {container["id"]: container["d"] for container in containers}
cid_to_w = {container["id"]: container["w"] for container in containers}
arrival_order = [container["id"] for container in containers]

assert len(containers) <= NUM_STACKS * MAX_TIERS, "The instance must fit within total stack capacity."

arrival_order

In [ ]:
# Blocking-pair metrics and local penalty signals.
# A pair is counted when a lower container departs before an upper container in the same stack.

def count_blocking_pairs_in_stack(
    stack_bottom_to_top: List[str],
    departure_times: Dict[str, int],
) -> int:
    """Count lower–upper pairs whose departure order creates blocking risk."""
    blocking_pairs = 0
    for lower_idx in range(len(stack_bottom_to_top)):
        for upper_idx in range(lower_idx + 1, len(stack_bottom_to_top)):
            lower = stack_bottom_to_top[lower_idx]
            upper = stack_bottom_to_top[upper_idx]
            if departure_times[lower] < departure_times[upper]:
                blocking_pairs += 1
    return blocking_pairs


def count_total_reshuffles(
    stack_config: List[List[str]],
    departure_times: Dict[str, int],
) -> int:
    """Return the total blocking-pair proxy across all stacks."""
    return sum(
        count_blocking_pairs_in_stack(stack, departure_times)
        for stack in stack_config
    )


def blocking_created_by_placing(
    stack: List[str],
    arriving_cid: str,
    departure_times: Dict[str, int],
) -> int:
    """Count new blocking pairs created by placing a container on this stack."""
    arriving_departure = departure_times[arriving_cid]
    return sum(
        departure_times[existing_cid] < arriving_departure
        for existing_cid in stack
    )


def weight_instability_created(
    stack: List[str],
    arriving_cid: str,
    weights: Dict[str, int],
) -> int:
    """Return 1 when the arriving container is heavier than the current top."""
    if not stack:
        return 0
    return int(weights[arriving_cid] > weights[stack[-1]])


def height_variance(heights: List[int]) -> float:
    """Measure stack-height imbalance using population variance."""
    return float(np.var(np.asarray(heights, dtype=float)))


# Sanity checks connect the pair definition to the helper functions.
assert count_total_reshuffles([["C2", "C1"], [], []], cid_to_d) == 1
assert blocking_created_by_placing(["C2"], "C1", cid_to_d) == 1
assert weight_instability_created(["C3"], "C4", cid_to_w) == 1

## Environment and state representation

A reinforcement-learning environment supplies observations, legal actions, rewards, and the next observation. Here an action is a stack index, and a stack at capacity is not a legal action.

The agent uses a feature vector containing stack heights, top-container departure times and weights, and information about the next arrival. This compact representation is easy to inspect, but it does not encode every container below the top. Consequently, two different stack configurations can have the same feature vector even if their future blocking risks differ. The learning problem is therefore an approximate-state version of the fully observed placement process.

### State, action, and transition

In reinforcement learning, a state summarizes information used to choose an action; an action is a decision available in that state; and a transition records the result of acting in the environment. Here the physical configuration is the list of stacks plus the index of the next arrival. The observation supplied to the linear model is a numeric encoding of that information, not the complete physical state.

If stack \(i\) has height \(h_i\), top departure time \(d_i^{\mathrm{top}}\), and top weight \(w_i^{\mathrm{top}}\), the encoded vector concatenates the normalized values for all stack heights, all top departure times, all top weights, and three arrival/progress features. Empty stacks use 0 for their top-container features. A full stack is excluded from the legal action set:

$$A(S_t)=\{i\in\{0,\ldots,N-1\}: |S_{i,t}|<M\}.$$

The implementation uses zero-based stack indices, so the formula's indices correspond to the values returned by `valid_actions`. `encode_state` constructs the observation, `valid_actions` constructs the legal action set, and `step_env` applies the placement and returns the next configuration, reward, and terminal flag. The compact observation may not retain the Markov property: identical feature vectors need not imply identical distributions of future rewards because hidden lower-tier containers can differ.

In [ ]:
# The feature vector has one group for stack heights, one for top departure times,
# one for top weights, and three features for the next arrival and episode progress.
max_d = max(container["d"] for container in containers)
max_w = max(container["w"] for container in containers)


def encode_state(
    stack_config: List[List[str]],
    arriving_idx: int,
    arrivals: List[str],
    departure_times: Dict[str, int],
    weights: Dict[str, int],
    max_departure: int,
    max_weight: int,
    max_tiers: int,
) -> np.ndarray:
    """Encode the current placement configuration as normalized numeric features."""
    heights = np.asarray(
        [len(stack) / max_tiers for stack in stack_config],
        dtype=np.float32,
    )

    top_departures = []
    top_weights = []
    for stack in stack_config:
        if not stack:
            top_departures.append(0.0)
            top_weights.append(0.0)
        else:
            top_cid = stack[-1]
            top_departures.append(departure_times[top_cid] / max_departure)
            top_weights.append(weights[top_cid] / max_weight)

    top_departures = np.asarray(top_departures, dtype=np.float32)
    top_weights = np.asarray(top_weights, dtype=np.float32)

    if arriving_idx >= len(arrivals):
        arriving_departure = 0.0
        arriving_weight = 0.0
        progress = 1.0
    else:
        arriving_cid = arrivals[arriving_idx]
        arriving_departure = departure_times[arriving_cid] / max_departure
        arriving_weight = weights[arriving_cid] / max_weight
        progress = arriving_idx / max(1, len(arrivals) - 1)

    arrival_features = np.asarray(
        [arriving_departure, arriving_weight, progress],
        dtype=np.float32,
    )
    return np.concatenate([heights, top_departures, top_weights, arrival_features])


empty_state = encode_state(
    [[], [], []], 0, arrival_order, cid_to_d, cid_to_w, max_d, max_w, MAX_TIERS
)
partly_filled_state = encode_state(
    [["C1"], [], []], 1, arrival_order, cid_to_d, cid_to_w, max_d, max_w, MAX_TIERS
)
empty_state, partly_filled_state

In [ ]:
# The environment returns a new configuration, the next arrival index, a reward,
# a terminal flag, and diagnostic information for the current placement.

def valid_actions(stack_config: List[List[str]], max_tiers: int) -> List[int]:
    """Return stack indices that still have capacity."""
    return [index for index, stack in enumerate(stack_config) if len(stack) < max_tiers]


def step_env(
    stack_config: List[List[str]],
    arriving_idx: int,
    action: int,
    arrivals: List[str],
    departure_times: Dict[str, int],
    weights: Dict[str, int],
    max_tiers: int,
    w_block: float = 5.0,
    w_height_var: float = 1.0,
    w_weight: float = 2.0,
    terminal_reshuffle_weight: float = 3.0,
):
    """Apply one placement and return (configuration, index, reward, done, info)."""
    if arriving_idx >= len(arrivals):
        return stack_config, arriving_idx, 0.0, True, {"done": True}

    if not 0 <= action < len(stack_config):
        raise ValueError("Action must be a valid stack index.")

    arriving_cid = arrivals[arriving_idx]
    info = {"arriving": arriving_cid, "action": int(action), "invalid": False}

    if len(stack_config[action]) >= max_tiers:
        info["invalid"] = True
        return stack_config, arriving_idx, -20.0, False, info

    block_created = blocking_created_by_placing(
        stack_config[action], arriving_cid, departure_times
    )
    weight_created = weight_instability_created(
        stack_config[action], arriving_cid, weights
    )

    new_stack_config = [stack.copy() for stack in stack_config]
    new_stack_config[action].append(arriving_cid)
    height_var = height_variance([len(stack) for stack in new_stack_config])

    reward = -(
        w_block * float(block_created)
        + w_height_var * height_var
        + w_weight * float(weight_created)
    )

    next_idx = arriving_idx + 1
    done = next_idx >= len(arrivals)
    if done:
        terminal_pairs = float(count_total_reshuffles(new_stack_config, departure_times))
        reward -= terminal_reshuffle_weight * terminal_pairs
        info["terminal_total_reshuffles"] = terminal_pairs

    info.update(
        {
            "block_created": int(block_created),
            "weight_instability": int(weight_created),
            "height_variance": float(height_var),
        }
    )
    return new_stack_config, next_idx, float(reward), bool(done), info


initial_config = [[], [], []]
config_after_first, index_after_first, reward_after_first, done_after_first, info_after_first = step_env(
    initial_config, 0, 0, arrival_order, cid_to_d, cid_to_w, MAX_TIERS
)
(config_after_first, index_after_first, reward_after_first, done_after_first, info_after_first)

## Linear Q-learning agent with experience replay

For a state feature vector \(s\) and stack action \(a\), the agent estimates an action value:

$$Q(s,a)=w_a^{\mathsf T}s+b_a.$$

Here, \(w_a\) is the weight vector and \(b_a\) is the bias for action \(a\). A larger value means a more promising expected discounted return. Since rewards are penalties, the agent seeks actions with higher (usually less negative) values.

### Q-learning target and linear update

The action-value function estimates the return expected after taking action \(a\) in state \(s\) and then following a policy. A temporal-difference (TD) update compares the current estimate with a one-step target. For a transition \((s,a,r,s')\), define

$$y=\begin{cases}
r, & \text{if the transition is terminal},\\
r+\gamma\displaystyle\max_{a'\in A(s')}Q(s',a'), & \text{otherwise},
\end{cases}
\qquad
\delta=y-Q(s,a).$$

Here \(r\) is the received reward, \(\gamma\) is the discount factor, \(A(s')\) is the set of legal actions in the next state, and \(\delta\) is the TD error. The terminal case has no future action value because the episode has ended. For the linear model, the semi-gradient update is

$$w_a\leftarrow w_a+\alpha\delta s,
\qquad
b_a\leftarrow b_a+\alpha\delta,$$

where \(\alpha\) is the learning rate. This update moves the estimate for the selected action toward the target while leaving the other action parameters unchanged. In `replay`, `target`, `td_error`, `self.W[action]`, and `self.b[action]` implement these quantities; invalid next actions are excluded from the maximum.

The replay buffer stores transitions and reuses randomly sampled experiences. This implementation uses a linear function approximator, epsilon-greedy exploration, and replay; it does not use a neural network or a separate target network. Experience replay changes the order in which stored transitions contribute updates; it does not change the placement environment or the reward definition. Because the observation is approximate and the model is trained from sampled experience, this implementation does not by itself provide a convergence or optimality guarantee.

In [ ]:
from collections import deque


class LinearDQN:
    """Linear action-value model trained with Q-learning and replay."""

    def __init__(
        self,
        state_size: int,
        action_size: int,
        lr: float = 0.08,
        gamma: float = 0.98,
    ):
        self.state_size = state_size
        self.action_size = action_size
        self.lr = lr
        self.gamma = gamma
        self.W = rng.normal(0, 0.1, size=(action_size, state_size)).astype(np.float32)
        self.b = np.zeros(action_size, dtype=np.float32)
        self.memory = deque(maxlen=10000)

    def q_values(self, state: np.ndarray) -> np.ndarray:
        """Compute estimated values for every stack action."""
        return (self.W @ state) + self.b

    def act(self, state: np.ndarray, valid_acts: List[int], eps: float) -> int:
        """Choose a legal action using epsilon-greedy exploration."""
        if not valid_acts:
            raise ValueError("At least one legal action is required before an episode ends.")

        if rng.random() < eps:
            return int(rng.choice(valid_acts))

        q = self.q_values(state)
        masked = np.full_like(q, -1e9)
        for action in valid_acts:
            masked[action] = q[action]
        return int(np.argmax(masked))

    def remember(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_valid_actions: List[int],
    ) -> None:
        self.memory.append(
            (state, action, reward, next_state, done, next_valid_actions)
        )

    def replay(self, batch_size: int = 64) -> float:
        """Apply semi-gradient TD updates to a random transition batch."""
        if len(self.memory) < batch_size:
            return 0.0

        indices = rng.choice(len(self.memory), size=batch_size, replace=False)
        squared_errors = []

        for index in indices:
            state, action, reward, next_state, done, next_valid_actions = self.memory[index]
            current_q = self.q_values(state)

            target = float(reward)
            if not done:
                if not next_valid_actions:
                    raise ValueError("A nonterminal transition must have a legal next action.")
                next_q = self.q_values(next_state)
                masked_next_q = np.full_like(next_q, -1e9)
                for next_action in next_valid_actions:
                    masked_next_q[next_action] = next_q[next_action]
                target += self.gamma * float(np.max(masked_next_q))

            td_error = target - float(current_q[action])
            self.W[action] += self.lr * td_error * state
            self.b[action] += self.lr * td_error
            squared_errors.append(td_error * td_error)

        return float(np.mean(squared_errors))


STATE_SIZE = int(
    encode_state(
        [[], [], []], 0, arrival_order, cid_to_d, cid_to_w, max_d, max_w, MAX_TIERS
    ).shape[0]
)
ACTION_SIZE = NUM_STACKS
agent = LinearDQN(STATE_SIZE, ACTION_SIZE)

STATE_SIZE, ACTION_SIZE

## Training loop

Each episode starts with empty stacks and processes the same arrival sequence. The agent selects actions, receives rewards, and stores transitions in its replay buffer. The exploration probability \(\epsilon\) gradually decreases: early episodes explore more, while later episodes rely more heavily on learned action values.

At each decision, epsilon-greedy selection chooses a random legal action with probability \(\epsilon\); otherwise, it chooses a legal action with the largest estimated action value. After a transition is stored, replay samples a batch without replacement from the buffer and applies TD updates. This sequence links the environment interaction to the update rule: observations and legal actions determine the decision, the environment returns a reward and next observation, and the stored transition later contributes to learning.

The recorded quantities are total episode reward, final blocking-pair count, and mean TD loss. A less negative reward is favorable under the chosen reward definition, but neither reward nor loss alone proves that the policy is optimal. The reshuffle metric and the later exact benchmark provide additional context.

In [ ]:
def run_episode(eps: float, train: bool = True, max_steps: int = 20):
    """Run one episode and return reward, blocking pairs, plan, log, and mean TD loss."""
    stack_config: List[List[str]] = [[] for _ in range(NUM_STACKS)]
    arriving_idx = 0
    total_reward = 0.0
    losses = []
    step_log = []

    for step in range(max_steps):
        state = encode_state(
            stack_config, arriving_idx, arrival_order, cid_to_d, cid_to_w,
            max_d, max_w, MAX_TIERS
        )
        actions = valid_actions(stack_config, MAX_TIERS)
        action = agent.act(state, actions, eps=eps)

        next_config, next_idx, reward, done, info = step_env(
            stack_config, arriving_idx, action, arrival_order, cid_to_d, cid_to_w,
            MAX_TIERS
        )
        total_reward += float(reward)

        next_state = encode_state(
            next_config, next_idx, arrival_order, cid_to_d, cid_to_w,
            max_d, max_w, MAX_TIERS
        )
        next_actions = valid_actions(next_config, MAX_TIERS)

        if train:
            agent.remember(state, action, reward, next_state, done, next_actions)
            losses.append(agent.replay(batch_size=64))

        step_log.append(
            {
                "t": step,
                "arriving": info.get("arriving"),
                "action_stack": int(action) + 1,
                "reward": float(reward),
                "block_created": info.get("block_created"),
                "weight_instability": info.get("weight_instability"),
                "height_variance": info.get("height_variance"),
                "done": bool(done),
            }
        )

        stack_config, arriving_idx = next_config, next_idx
        if done:
            break

    if arriving_idx < len(arrival_order):
        raise RuntimeError("The episode reached max_steps before all containers were placed.")

    terminal_pairs = count_total_reshuffles(stack_config, cid_to_d)
    mean_loss = float(np.mean(losses)) if losses else 0.0
    return (
        float(total_reward),
        terminal_pairs,
        stack_config,
        pd.DataFrame(step_log),
        mean_loss,
    )


run_episode(eps=1.0, train=False)[:2]

In [ ]:
# Decaying epsilon balances early exploration with later greedy action selection.
episodes = 1200
eps0 = 1.0
eps_min = 0.05
decay = 0.995

ep_rewards = []
ep_reshuffles = []
ep_losses = []

eps = eps0
for episode in range(episodes):
    total_reward, terminal_pairs, final_config, step_df, loss = run_episode(
        eps=eps, train=True
    )
    ep_rewards.append(total_reward)
    ep_reshuffles.append(terminal_pairs)
    ep_losses.append(loss)
    eps = max(eps_min, eps * decay)

print("Training finished.")
print(f"Final epsilon: {eps:.3f}")
print(f"Mean blocking pairs over the last 50 episodes: {np.mean(ep_reshuffles[-50:]):.3f}")
print(f"Mean reward over the last 50 episodes: {np.mean(ep_rewards[-50:]):.3f}")

## Training curves

The plots show episode reward, final blocking-pair count, and mean TD loss. These are diagnostics rather than guarantees of convergence. Reward and blocking-pair trends can be noisy because the policy explores, while a declining TD loss does not by itself imply a good stacking plan.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

axes[0].plot(ep_rewards, lw=1)
axes[0].set_title("Episode total reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward (penalty units)")
axes[0].grid(True, alpha=0.25)

axes[1].plot(ep_reshuffles, lw=1)
axes[1].set_title("Blocking pairs per episode")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Blocking-pair count")
axes[1].grid(True, alpha=0.25)

axes[2].plot(ep_losses, lw=1)
axes[2].set_title("Mean TD loss per episode")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Mean squared TD error")
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## Greedy evaluation after training

Setting \(\epsilon=0\) removes random exploration: at each step, the agent selects a legal stack with the highest estimated action value. This is one greedy evaluation episode on the same fixed arrival order, not a statistical estimate of performance across different instances.

The step log records local reward components, while the final configuration and blocking-pair count summarize the resulting plan. If multiple legal actions have equal estimated values, `numpy.argmax` selects the first one in the action array; this tie rule is an implementation detail and does not establish that the corresponding physical plan is uniquely best.

In [ ]:
greedy_reward, greedy_resh, greedy_cfg, greedy_steps, _ = run_episode(
    eps=0.0, train=False
)

print("Greedy total reward:", greedy_reward)
print("Greedy terminal blocking pairs:", greedy_resh)
print("Greedy final configuration:", greedy_cfg)

greedy_steps

## Exact small-instance benchmark

For this small instance, every sequence of stack choices can be enumerated and infeasible sequences that exceed stack capacity can be discarded. The minimum blocking-pair count among feasible plans is an exact benchmark for that metric. It is not an exact optimization of the full RL reward, which also includes height-variance and weight-instability penalties.

If several plans attain the same minimum, the benchmark reports their shared objective value; it does not imply that the minimizing plan is unique. The exhaustive search has \(N^T\) candidate action sequences before infeasible sequences are filtered, where \(N\) is the number of stacks and \(T\) is the number of arriving containers. For each candidate, constructing the plan takes \(O(T)\) operations, and counting blocking pairs takes at most \(O(T^2)\) pair comparisons across the stacks. Thus the simple exhaustive benchmark has overall order \(O(N^T T^2)\) time in this implementation and stores the candidate sequence list as well as evaluating its elements.

The benchmark is a separate evaluation of the final blocking-pair metric, not part of the agent's training update. Comparing the greedy policy with this minimum shows the gap for this one instance and metric; it does not certify optimality for the combined reward or for other instances.

In [ ]:
import itertools


def apply_choice_sequence(
    choice_seq: Tuple[int, ...],
    arrivals: List[str],
    num_stacks: int,
    max_tiers: int,
) -> Optional[List[List[str]]]:
    """Build a stack configuration, or return None if a choice exceeds capacity."""
    stacks: List[List[str]] = [[] for _ in range(num_stacks)]
    for cid, stack_idx in zip(arrivals, choice_seq):
        if len(stacks[stack_idx]) >= max_tiers:
            return None
        stacks[stack_idx].append(cid)
    return stacks


all_choice_sequences = list(
    itertools.product(range(NUM_STACKS), repeat=len(arrival_order))
)
opt_resh = None

for choice_seq in all_choice_sequences:
    config = apply_choice_sequence(choice_seq, arrival_order, NUM_STACKS, MAX_TIERS)
    if config is None:
        continue
    blocking_pairs_count = count_total_reshuffles(config, cid_to_d)
    if opt_resh is None or blocking_pairs_count < opt_resh:
        opt_resh = blocking_pairs_count

assert opt_resh is not None, "At least one feasible stacking plan must exist."
assert greedy_resh >= opt_resh, "The evaluated feasible plan cannot beat the exact minimum."

print("Minimum blocking-pair count (exact):", int(opt_resh))
print("Greedy policy blocking-pair count:", int(greedy_resh))
print("Gap (greedy policy minus exact minimum):", int(greedy_resh) - int(opt_resh))

In [ ]:
def blocking_pairs(
    stack_config: List[List[str]],
    departure_times: Dict[str, int],
) -> pd.DataFrame:
    """List each lower–upper pair counted by the blocking-risk metric."""
    pairs = []
    for stack_idx, stack in enumerate(stack_config, start=1):
        for lower_idx in range(len(stack)):
            for upper_idx in range(lower_idx + 1, len(stack)):
                lower = stack[lower_idx]
                upper = stack[upper_idx]
                if departure_times[lower] < departure_times[upper]:
                    pairs.append(
                        {
                            "stack": stack_idx,
                            "lower": lower,
                            "upper": upper,
                            "lower_departure": departure_times[lower],
                            "upper_departure": departure_times[upper],
                        }
                    )
    return pd.DataFrame(
        pairs,
        columns=["stack", "lower", "upper", "lower_departure", "upper_departure"],
    )


blocking_df = blocking_pairs(greedy_cfg, cid_to_d)
assert len(blocking_df) == greedy_resh

# Draw stack slots first so both occupied and empty capacity are visible.
fig, ax = plt.subplots(figsize=(8, 4))
stack_width = 1.0
cell_height = 1.0
x_gap = 0.6
stack_x = {
    stack_idx: (stack_idx - 1) * (stack_width + x_gap)
    for stack_idx in range(1, NUM_STACKS + 1)
}

for stack_idx in range(1, NUM_STACKS + 1):
    x0 = stack_x[stack_idx]
    for tier in range(MAX_TIERS):
        ax.add_patch(
            plt.Rectangle(
                (x0, tier * cell_height),
                stack_width,
                cell_height,
                facecolor="#F2F4F7",
                edgecolor="#344054",
            )
        )

for stack_idx, stack in enumerate(greedy_cfg, start=1):
    x0 = stack_x[stack_idx]
    for tier, cid in enumerate(stack):
        ax.text(
            x0 + stack_width / 2,
            tier * cell_height + cell_height / 2,
            f"{cid}\n(d={cid_to_d[cid]}, w={cid_to_w[cid]})",
            ha="center",
            va="center",
            fontsize=9,
            color="#101828",
        )

for _, row in blocking_df.iterrows():
    stack_idx = int(row["stack"])
    stack = greedy_cfg[stack_idx - 1]
    lower_tier = stack.index(row["lower"])
    upper_tier = stack.index(row["upper"])
    x = stack_x[stack_idx] + stack_width / 2
    y_lower = lower_tier * cell_height + cell_height / 2
    y_upper = upper_tier * cell_height + cell_height / 2
    ax.annotate(
        "",
        xy=(x, y_lower),
        xytext=(x, y_upper),
        arrowprops={"arrowstyle": "->", "lw": 2, "color": "#F97066"},
    )

ax.set_title("Greedy policy stacking plan and blocking pairs")
ax.set_xlabel("Stack")
ax.set_ylabel("Tier (bottom to top)")
ax.set_xticks(
    [stack_x[stack_idx] + stack_width / 2 for stack_idx in range(1, NUM_STACKS + 1)]
)
ax.set_xticklabels([f"Stack {stack_idx}" for stack_idx in range(1, NUM_STACKS + 1)])
ax.set_yticks([tier * cell_height + cell_height / 2 for tier in range(MAX_TIERS)])
ax.set_yticklabels([f"Tier {tier + 1}" for tier in range(MAX_TIERS)])
ax.set_xlim(-0.5, stack_x[NUM_STACKS] + stack_width + 0.5)
ax.set_ylim(-0.2, MAX_TIERS * cell_height + 0.2)
plt.tight_layout()
plt.show()

blocking_df

In [ ]:
# Attribute each counted pair to the lower container that is blocked during retrieval.
block_counts = {container["id"]: 0 for container in containers}
for _, row in blocking_df.iterrows():
    block_counts[row["lower"]] += 1

retrieve_order = sorted(arrival_order, key=lambda cid: cid_to_d[cid])
reshuffles_per_container = [block_counts[cid] for cid in retrieve_order]

plt.figure(figsize=(7, 3.2))
plt.bar(
    retrieve_order,
    reshuffles_per_container,
    color="#2E90FA",
    edgecolor="#101828",
    alpha=0.85,
)
plt.title("Blocking-pair count attributed to each retrieved container")
plt.xlabel("Container (retrieval order)")
plt.ylabel("Blocking-pair count")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

cumulative_pairs = np.cumsum(reshuffles_per_container)
plt.figure(figsize=(7, 3.2))
plt.plot(retrieve_order, cumulative_pairs, marker="o", lw=2, color="#344054")
plt.title("Cumulative blocking-pair count in retrieval order")
plt.xlabel("Container (retrieval order)")
plt.ylabel("Cumulative blocking-pair count")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        "container": retrieve_order,
        "departure_time": [cid_to_d[cid] for cid in retrieve_order],
        "blocking_pairs": reshuffles_per_container,
    }
)

## When a learned policy is useful

Search and optimization methods construct a plan for a specified instance. A learned policy instead maps observed features to actions and may be useful when many related decisions must be made repeatedly or decisions arrive sequentially. Whether it generalizes depends on the training instances, state representation, reward design, and evaluation; this fixed-instance exercise does not test generalization.

The approach requires training and can be sensitive to reward weights and learning settings. Unlike an exact benchmark on a small instance, a learned policy does not automatically provide an optimality guarantee. Its quality should be assessed using operationally meaningful metrics and a suitable range of test cases.

## Modeling and training considerations

The reward weights determine the relative importance of blocking risk, height variance, and the weight-instability signal. Changing them changes the learning objective, so they should be selected to reflect the intended application rather than to obtain a preferred training curve. Training can also be sensitive to the number of episodes, learning rate, and exploration schedule. In this example, legal actions are filtered before selection, so a full-stack action is not chosen by the policy.

## Further extensions

The same learning framework can be studied on additional arrival sequences and container attributes, with performance evaluated on instances distinct from those used for training. A richer state representation could retain more stack information. These are possible extensions; the implementation here remains the stated linear Q-learning approach on the fixed instance.